In [ ]:
# ==============================================================================
# The code in the main text
# ==============================================================================

import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import IsolationForest

# Models
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

# Explainability
import shap
from geoshapley import GeoShapleyExplainer

warnings.filterwarnings('ignore')

# ==========================================
# 1. Configuration & Setup
# ==========================================
def set_all_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_all_seeds(42)
sns.set_style("white")

# Create directories for saving figures
os.makedirs('model_results', exist_ok=True)
os.makedirs('shap_figures', exist_ok=True)

# ==========================================
# 2. Data Loading & Preprocessing
# ==========================================
# Placeholder for data loading
DATA_PATH = "path_to_your_dataset.csv" # TODO: Replace with generic dataset name if providing sample data

try:
    data = pd.read_csv(DATA_PATH)
except FileNotFoundError:
    # Creating dummy data structure for demonstration purposes
    print("Dataset not found. Please place the data in the correct path.")
    # data = pd.DataFrame(...) 

# Handle missing values (Generic approach)
for col in data.columns:
    if pd.api.types.is_numeric_dtype(data[col]):
        data[col] = data[col].fillna(data[col].mean())

# Define features and target
target = 'composite urban vitality'
# Assuming coordinates are named 'UTM_X' and 'UTM_Y'
geo_features = ['UTM_X', 'UTM_Y']
features = [col for col in data.columns if col != target]

# Scale coordinates
scaler = MinMaxScaler()
data[geo_features] = scaler.fit_transform(data[geo_features])

X = data[features]
y = data[target]

# Optional: Outlier removal function (Skeleton)
def remove_outliers(df, feature_cols, contamination=0.1):
    iso_forest = IsolationForest(contamination=contamination, random_state=42)
    outliers = iso_forest.fit_predict(df[feature_cols])
    return df[outliers == 1]

# data = remove_outliers(data, features)

# ==========================================
# 3. Model Definition & Cross Validation
# ==========================================
# Initialize models with default or baseline parameters to avoid leaking optimal hyperparameters
models = {
    'RandomForest': RandomForestRegressor(random_state=42),
    'XGBoost': XGBRegressor(random_state=42),
    'CatBoost': CatBoostRegressor(verbose=False, random_state=42),
    'LightGBM': LGBMRegressor(verbose=-1, random_state=42)
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

print("Starting Cross-Validation...")
for model_name, model in models.items():
    r2_scores, mae_scores, rmse_scores = [], [], []
    
    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        
        r2_scores.append(r2_score(y_val, y_pred))
        mae_scores.append(mean_absolute_error(y_val, y_pred))
        rmse_scores.append(np.sqrt(mean_squared_error(y_val, y_pred)))
        
    cv_results.append({
        'Model': model_name,
        'R2_Mean': np.mean(r2_scores),
        'MAE_Mean': np.mean(mae_scores),
        'RMSE_Mean': np.mean(rmse_scores)
    })

cv_df = pd.DataFrame(cv_results)
cv_df.to_csv('model_results/cv_summary.csv', index=False)
print("Cross-Validation completed. Results saved.")

# ==========================================
# 4. SHAP Analysis (Global & Local)
# ==========================================
# Select a primary model for interpretation (e.g., CatBoost or XGBoost)
best_model = models['CatBoost'] 
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(X, y, test_size=0.2, random_state=42)
best_model.fit(X_train_full, y_train_full)

print("Generating SHAP explanations...")
explainer = shap.Explainer(best_model.predict, X_test_full)
shap_values = explainer(X_test_full)

# 4.1 SHAP Summary Plot (Bar)
shap.summary_plot(shap_values, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig('shap_figures/shap_summary_bar.pdf', dpi=300, bbox_inches='tight')
plt.close()

# 4.2 SHAP Summary Plot (Dot/Scatter)
shap.summary_plot(shap_values, show=False)
plt.tight_layout()
plt.savefig('shap_figures/shap_summary_scatter.pdf', dpi=300, bbox_inches='tight')
plt.close()

# 4.3 SHAP Dependence Plots
for feature in features[:3]: # Just plotting first few to save time in skeleton
    shap.dependence_plot(feature, shap_values.values, X_test_full, interaction_index='auto', show=False)
    plt.tight_layout()
    plt.savefig(f'shap_figures/shap_dependence_{feature}.pdf', dpi=300, bbox_inches='tight')
    plt.close()

# ==========================================
# 5. GeoShapley Spatial Analysis
# ==========================================
print("Starting GeoShapley Analysis...")
# Sample background data for GeoShapley to reduce computation time
background_data = X_train_full.sample(n=min(50, len(X_train_full)), random_state=42).values

# Initialize GeoShapley Explainer
geo_explainer = GeoShapleyExplainer(best_model.predict, background_data)

# Explain the test set (or full dataset)
geo_result = geo_explainer.explain(X_test_full)

# 5.1 GeoShapley Summary Plot
geo_result.summary_plot()
plt.savefig("shap_figures/geo_shap_summary.pdf", format='pdf', bbox_inches='tight', dpi=300)
plt.close()

# 5.2 Extract and Save GeoShapley Values
geo_summary_df = geo_result.summary_statistics()
geo_summary_df.to_csv('shap_figures/geoshapley_statistics.csv')

# Combine original data with GeoShapley values for GIS mapping
geoshap_values = geo_result.primary
geoshap_df = pd.DataFrame(geoshap_values, columns=[f'geoshap_{col}' for col in features])

# Inverse transform coordinates for mapping
X_test_mapping = X_test_full.copy()
X_test_mapping[geo_features] = scaler.inverse_transform(X_test_mapping[geo_features])

# Merge and save
final_export_df = pd.concat([X_test_mapping.reset_index(drop=True), geoshap_df.reset_index(drop=True)], axis=1)
final_export_df.to_csv('shap_figures/spatial_mapping_data.csv', index=False)

print("Pipeline executed successfully. All outputs saved.")